# Exploratory Data Analysis — Predicting Stellar Class (S6E6)

**Competition:** https://www.kaggle.com/competitions/playground-series-s6e6  
**Goal:** Classify astronomical objects as GALAXY, STAR, or QSO  
**Metric:** Balanced Accuracy

### Contents
1. Load data
2. Dataset overview
3. Target distribution
4. Feature distributions
5. Correlation analysis
6. Class-conditional distributions
7. Key takeaways

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

import sys
sys.path.append('..')
from src.data import load_data

train, test = load_data('../data/')
print('Train:', train.shape)
print('Test: ', test.shape)

## 1. Dataset Overview

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
# Missing values
missing = train.isnull().sum()
print('Missing values in train:')
print(missing[missing > 0] if missing.sum() > 0 else 'None')

## 2. Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = train['class'].value_counts()
counts.plot(kind='bar', ax=ax, color=['steelblue', 'darkorange', 'green'])
ax.set_title('Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
for i, v in enumerate(counts):
    ax.text(i, v + 50, f'{v:,} ({v/len(train)*100:.1f}%)', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
photometric = ['u', 'g', 'r', 'i', 'z', 'redshift']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), photometric):
    for cls, color in zip(['GALAXY', 'STAR', 'QSO'], ['steelblue', 'darkorange', 'green']):
        data = train.loc[train['class'] == cls, col].dropna()
        ax.hist(data, bins=50, alpha=0.5, label=cls, color=color, density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Photometric Feature Distributions by Class', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
numeric_cols = train.select_dtypes(include='number').columns.tolist()
corr = train[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.3)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Redshift vs Photometric Bands

In [ ]:
sample = train.sample(min(5000, len(train)), random_state=42)
palette = {'GALAXY': 'steelblue', 'STAR': 'darkorange', 'QSO': 'green'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, color in palette.items():
    sub = sample[sample['class'] == cls]
    axes[0].scatter(sub['redshift'], sub['r'], alpha=0.3, s=5, color=color, label=cls)
    axes[1].scatter(sub['g'] - sub['r'], sub['r'] - sub['i'], alpha=0.3, s=5, color=color, label=cls)

axes[0].set_xlabel('Redshift')
axes[0].set_ylabel('r magnitude')
axes[0].set_title('Redshift vs r magnitude')
axes[0].legend()

axes[1].set_xlabel('g - r (colour)')
axes[1].set_ylabel('r - i (colour)')
axes[1].set_title('Colour–Colour Diagram')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Key Takeaways

- **Redshift** is the single most discriminating feature: QSOs cluster at high redshifts, stars near zero.
- **Photometric colours** (band differences) are more informative than raw magnitudes.
- The dataset appears relatively balanced; balanced accuracy is still the evaluation metric so all classes matter.
- `alpha` / `delta` (sky coordinates) show little class separation and can be dropped.
- ID columns (`obj_ID`, `run_ID`, etc.) carry no signal and should be dropped.